# Task 76: Accessible Custom-CSS Dashboard

Create a responsive, keyboard-aware prediction review interface with manually authored CSS, readable contrast, reduced-motion support, and confidence-aware explanation text.

In [1]:
from pathlib import Path
from IPython.display import HTML, display
import html
import numpy as np
import pandas as pd

paths=[Path('heart.csv'),Path('Task_76_Accessible_Custom_CSS_Dashboard/heart.csv'),Path('Heart_Disease_Tasks/Task_76_Accessible_Custom_CSS_Dashboard/heart.csv')]
data=pd.read_csv(next(p for p in paths if p.is_file())).drop_duplicates().reset_index(drop=True)
features=[c for c in ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal'] if c in data]
X,y=data[features].to_numpy(float),data.target.to_numpy(int)
rng=np.random.default_rng(76);train=[];test=[]
for label in np.unique(y):
    ids=np.flatnonzero(y==label);rng.shuffle(ids);cut=round(.8*len(ids));train.extend(ids[:cut]);test.extend(ids[cut:])
train,test=np.array(train),np.array(test);rng.shuffle(train);rng.shuffle(test)
Xtr,Xte=X[train],X[test];ytr,yte=y[train],y[test]
mean,scale=Xtr.mean(0),Xtr.std(0);scale[scale==0]=1;Xtr,Xte=(Xtr-mean)/scale,(Xte-mean)/scale
w=np.zeros(Xtr.shape[1]);b=0.
for _ in range(1400):
    p=1/(1+np.exp(-np.clip(Xtr@w+b,-35,35)));e=p-ytr;w-=.06*(Xtr.T@e/len(ytr)+.02*w);b-=.06*e.mean()
case=0;values=Xte[case];prob=float(1/(1+np.exp(-np.clip(values@w+b,-35,35))));prediction='Positive' if prob>=.5 else 'Negative';confidence=max(prob,1-prob)
contribution=w*values;top=np.argsort(np.abs(contribution))[::-1][:3]
reason=', '.join(f'{html.escape(features[i])} ({contribution[i]:+.2f} logit)' for i in top)
confidence_label='Higher' if confidence>=.8 else 'Moderate' if confidence>=.65 else 'Low'
page=f'''<style>
:root{{color-scheme:light}}body{{font-family:Arial,sans-serif;background:#f3f6f7;color:#172a35;margin:0;padding:16px}}
.hero{{background:linear-gradient(120deg,#123b4a,#176b68);color:white;padding:22px;border-radius:10px;animation:enter .4s ease-out}}
.grid{{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:12px;margin:14px 0}}
.card{{background:white;border:1px solid #d5e0e3;border-top:4px solid #1c7972;border-radius:8px;padding:15px;transition:transform .18s ease,box-shadow .18s ease}}
.card:hover{{transform:translateY(-2px);box-shadow:0 6px 16px #123b4a22}}
.explain{{background:#e9f4f2;border-left:4px solid #176b68;padding:14px;border-radius:4px;line-height:1.55}}
.label{{color:#425b64;font-size:14px}}.value{{font-size:25px;font-weight:bold;color:#123b4a;margin-top:6px}}
button{{background:#135c63;color:white;border:2px solid #135c63;border-radius:6px;padding:10px 16px;min-height:44px;cursor:pointer;transition:background .15s ease}}
button:hover{{background:#0c4349}}button:focus-visible{{outline:3px solid #d68137;outline-offset:2px}}
@keyframes enter{{from{{opacity:0;transform:translateY(8px)}}to{{opacity:1;transform:translateY(0)}}}}
@media(max-width:700px){{.grid{{grid-template-columns:1fr}}.hero{{padding:16px}}}}
@media(prefers-reduced-motion:reduce){{*,*::before,*::after{{animation-duration:.01ms!important;transition-duration:.01ms!important}}}}
</style>
<section class="hero"><h1>Heart Model Review</h1><p>Interactive prediction review and contribution context</p></section>
<div class="explain">Model predicts the {prediction.lower()} class with {confidence_label.lower()} confidence. Largest score associations: {reason}. These are model associations, not causes.</div>
<section class="grid"><div class="card"><div class="label">Predicted class</div><div class="value">{prediction}</div></div><div class="card"><div class="label">Positive-class score</div><div class="value">{prob:.1%}</div></div><div class="card"><div class="label">Model confidence</div><div class="value">{confidence:.1%}</div></div></section>
<p class="label">Educational example only. Confidence describes model output, not certainty of a clinical outcome.</p>'''
display(HTML(page))
contributions=pd.DataFrame({'Feature':features,'Standardized value':values,'Logit contribution':contribution}).sort_values('Logit contribution',key=np.abs,ascending=False)
display(contributions.head(8).round(3))
print('Responsive breakpoints, keyboard focus styling, hover transitions, and reduced-motion preferences are included in the custom CSS.')

,Feature,Standardized value,Logit contribution
11,ca,1.399,-1.491
7,thalach,-2.228,-0.743
8,exang,1.432,-0.667
12,thal,1.095,-0.570
2,cp,-0.949,-0.532
1,sex,0.659,-0.408
6,restecg,0.982,0.275
9,oldpeak,0.614,-0.259


Responsive breakpoints, keyboard focus styling, hover transitions, and reduced-motion preferences are included in the custom CSS.
